# CLIMP reproduction (CS613) — Kaggle runner
Thin runner: all logic lives in `src/`. Settings: **Accelerator = GPU T4 x2, Internet = On**, secret `HF_TOKEN` attached.
Set `MODE` / `STAGE` in cell 2, then **Save Version → Save & Run All**. Download `/kaggle/working/results_*.zip` from the Output tab.

In [ ]:
!nvidia-smi
!df -h /tmp /kaggle/working | cat

In [ ]:
import os, subprocess, sys, time

MODE = "smoke"   # "smoke" or "full"
STAGE = "A"      # full runs only: "A" = E1 E2 E6 E3 E4, "B" = E5 E7 E8
E8_EPOCHS = 25   # set from the smoke timing before Stage B

REPO_URL = "https://github.com/tanishwanve-git/climp-cs613.git"
UPSTREAM_COMMIT = "3ff1279bb87f905596e6187f0e4a14f2cd28e818"
REPO = "/kaggle/working/climp-cs613"
os.environ["DATA_DIR"] = "/tmp/data"            # datasets + HF cache stay out of /kaggle/working (20 GB limit)
os.environ["HF_HOME"] = "/tmp/data/hf"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as e:
    print("no HF_TOKEN secret (downloads may be rate-limited):", e)

In [ ]:
t0 = time.time()
!rm -rf {REPO} && git clone -q {REPO_URL} {REPO}
!git clone -q https://github.com/NimrodShabtay/CLIMP.git {REPO}/CLIMP && git -C {REPO}/CLIMP checkout -q {UPSTREAM_COMMIT}
!echo "repo: $(git -C {REPO} rev-parse --short HEAD)  upstream: $(git -C {REPO}/CLIMP rev-parse --short HEAD)"
# upstream requirements minus torch/torchvision/triton/mamba packages (keep Kaggle's CUDA build of torch)
!grep -viE "^(torch|torchvision|triton|causal-conv1d|mamba-ssm)" {REPO}/CLIMP/requirements.txt > /tmp/upstream_reqs.txt
!pip install -q -r /tmp/upstream_reqs.txt -r {REPO}/requirements.txt
print(f"base install: {time.time() - t0:.0f}s")
t1 = time.time()
!pip install -q causal-conv1d mamba-ssm --no-build-isolation
print(f"causal-conv1d + mamba-ssm install: {time.time() - t1:.0f}s  (if > ~900s, build a wheel once and keep it as a Kaggle Dataset)")
!python -c "import torch, transformers, mamba_ssm, selective_scan_cuda; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| transformers', transformers.__version__, '| mamba_ssm', mamba_ssm.__version__, '| selective_scan_cuda OK')"

In [ ]:
failures = []

def run(cmd, stop):
    """Run a command inside the repo, streaming its output. stop=True raises on failure (full runs)."""
    print(f"\n{'=' * 100}\n$ {cmd}", flush=True)
    t = time.time()
    get_ipython().system(f"cd {REPO} && {cmd}")
    code = get_ipython().user_ns.get("_exit_code", 0)
    print(f"--> exit code {code} after {time.time() - t:.0f}s", flush=True)
    if code:
        failures.append(cmd)
        if stop:
            raise RuntimeError(f"failed: {cmd}")

E8 = f"python src/exp_cifar_bias.py --epochs {E8_EPOCHS} --seeds 0 1 2"
JOBS = {
    "smoke": [
        "python -m pytest tests -q",
        "python src/exp_nocaps.py --smoke",
        "python src/exp_geometry.py --smoke",
        "python src/exp_efficiency.py --smoke",
        "python src/exp_zeroshot.py --smoke --shuffle-grid 1 4",
        "python src/exp_docci.py --smoke",
        "python src/exp_sugarcrepe.py --smoke",
        "python src/exp_cifar_bias.py --smoke",
    ],
    "A": [
        "python src/exp_nocaps.py",                                        # E1 (exits non-zero if the README gate fails)
        "rm -rf $HF_HOME/hub/models--AntonV--mamba2-1.3b-hf $HF_HOME/hub/models--NimrodShabtay1986--CLIMP-Mamba2",
        "python src/exp_geometry.py",                                      # E2
        "python src/exp_efficiency.py",                                    # E6
        "python src/exp_zeroshot.py",                                      # E3
        "python src/exp_zeroshot.py --datasets imagenetv2 --shuffle-grid 2 4 8",  # E4 (G=1 comes from E3)
    ],
    "B": [
        "python src/exp_docci.py",                                         # E5
        "python src/exp_sugarcrepe.py",                                    # E7
        # E8: one model per T4, in parallel
        f"(CUDA_VISIBLE_DEVICES=0 {E8} --models vmamba > e8_vmamba.log 2>&1 & "
        f"CUDA_VISIBLE_DEVICES=1 {E8} --models vit > e8_vit.log 2>&1 & wait); tail -n 30 e8_vmamba.log e8_vit.log; "
        "! grep -l Traceback e8_vmamba.log e8_vit.log",
    ],
}
for cmd in JOBS["smoke" if MODE == "smoke" else STAGE]:
    run(cmd, stop=MODE == "full")
print("\nFAILED:" if failures else "\nALL COMMANDS SUCCEEDED", *failures, sep="\n  ")

In [ ]:
smoke = MODE == "smoke"
run("python src/make_results.py" + (" --smoke" if smoke else ""), stop=False)
name = "results_smoke" if smoke else f"results_{STAGE}"
folder = "results_smoke" if smoke else "results"
!cd {REPO} && rm -f /kaggle/working/{name}.zip && zip -qr /kaggle/working/{name}.zip {folder} -x "*/emb/*" && ls -lh /kaggle/working/{name}.zip